In [ ]:
# ============================================================
# Import libraries
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    XLMRobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# Set random seeds
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

In [ ]:
# ============================================================
# Experiment configuration
# ============================================================

MODEL_NAME = "FacebookAI/xlm-roberta-large" # ['xlm-roberta-base', 'xlm-roberta-large']

# Change these according to your dataset.
TEXT_COLUMN = "noisy_text"
LABEL_COLUMN = "label"

# Number of classification classes.
NUM_LABELS = 2

MAX_LENGTH = 128

# Checkpoints saving directory
OUTPUT_DIR = ""

# Trained model saving directory
FINAL_MODEL_DIR = ""

# Results saving directory (should end with .csv)
RESULTS_PATH = ""

print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_LENGTH)
print("Number of labels:", NUM_LABELS)

In [ ]:
# ============================================================
# Load JSONL dataset
# ============================================================

# Train and test datasets of the domain the model is fine-tuned on
TRAIN_PATH = ""
TEST_PATH = ""

# Test datasets of the domains the model is evaluated on
TEST_CLEAN_SINHALA = ""
TEST_TRANSLITERATION = ""
TEST_CODE_MIX = ""
TEST_MISSPELL = ""
TEST_COMPOUND = ""

train_df = pd.read_json(TRAIN_PATH, lines=True)
test_df = pd.read_json(TEST_PATH, lines=True)

test_transliteration_df = pd.read_json(TEST_TRANSLITERATION, lines=True)
test_codemix_df = pd.read_json(TEST_CODE_MIX, lines=True)
test_clean_sinhala_df = pd.read_json(TEST_CLEAN_SINHALA, lines=True)
test_misspell_df = pd.read_json(TEST_MISSPELL, lines=True)
test_compound_df = pd.read_json(TEST_COMPOUND, lines=True)

if "clean-sinhala-train-and-test" in TRAIN_PATH: 
    train_df = train_df.rename(columns={"text": TEXT_COLUMN})
    test_df = test_df.rename(columns={"text": TEXT_COLUMN})
test_clean_sinhala_df = test_clean_sinhala_df.rename(columns={"text": TEXT_COLUMN})

# ------------------------------------------------------------
# Split the ORIGINAL TRAINING SET into:
#
#   90% -> training
#   10% -> validation
#
# The existing TEST SET is NOT touched.
# ------------------------------------------------------------
train_df, valid_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df[LABEL_COLUMN]
)

# Reset dataframe indices after splitting.
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nAfter splitting:")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

# Check class distributions.
print("\nTrain label distribution:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nValidation label distribution:")
print(valid_df[LABEL_COLUMN].value_counts())

print("\nTest label distribution:")
print(test_df[LABEL_COLUMN].value_counts())

display(train_df.head())

In [ ]:
# ============================================================
# Inspect dataset
# ============================================================

print("Columns:")
print(train_df.columns.tolist())

print("\nFirst few samples:")
display(train_df.head())

print("\nLabel distribution:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nLabel distribution (%):")
print(train_df[LABEL_COLUMN].value_counts(normalize=True) * 100)

In [ ]:
# ============================================================
# Keep only text and label columns
# ============================================================

train_df = train_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
valid_df = valid_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_df = test_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

test_transliteration_df = test_transliteration_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_codemix_df = test_codemix_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_clean_sinhala_df = test_clean_sinhala_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_misspell_df = test_misspell_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_compound_df = test_compound_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

# Remove missing values.
train_df = train_df.dropna()
valid_df = valid_df.dropna()
test_df = test_df.dropna()

test_transliteration_df = test_transliteration_df.dropna()
test_codemix_df = test_codemix_df.dropna()
test_clean_sinhala_df = test_clean_sinhala_df.dropna()
test_misspell_df = test_misspell_df.dropna()
test_compound_df = test_compound_df.dropna()

# Make sure text is a string.
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)
valid_df[TEXT_COLUMN] = valid_df[TEXT_COLUMN].astype(str)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].astype(str)

test_transliteration_df[TEXT_COLUMN] = test_transliteration_df[TEXT_COLUMN].astype(str)
test_codemix_df[TEXT_COLUMN] = test_codemix_df[TEXT_COLUMN].astype(str)
test_clean_sinhala_df[TEXT_COLUMN] = test_clean_sinhala_df[TEXT_COLUMN].astype(str)
test_misspell_df[TEXT_COLUMN] = test_misspell_df[TEXT_COLUMN].astype(str)
test_compound_df[TEXT_COLUMN] = test_compound_df[TEXT_COLUMN].astype(str)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

In [ ]:
# ============================================================
# Encode labels
# ============================================================

# Convert NumPy integer labels to normal Python integers.
train_df[LABEL_COLUMN] = train_df[LABEL_COLUMN].astype(int)
valid_df[LABEL_COLUMN] = valid_df[LABEL_COLUMN].astype(int)
test_df[LABEL_COLUMN] = test_df[LABEL_COLUMN].astype(int)

test_transliteration_df[LABEL_COLUMN] = test_transliteration_df[LABEL_COLUMN].astype(int)
test_codemix_df[LABEL_COLUMN] = test_codemix_df[LABEL_COLUMN].astype(int)
test_clean_sinhala_df[LABEL_COLUMN] = test_clean_sinhala_df[LABEL_COLUMN].astype(int)
test_misspell_df[LABEL_COLUMN] = test_misspell_df[LABEL_COLUMN].astype(int)
test_compound_df[LABEL_COLUMN] = test_compound_df[LABEL_COLUMN].astype(int)

# Define the mapping explicitly.
label2id = {
    "non-hate": 0,
    "hate": 1,
}

id2label = {
    0: "non-hate",
    1: "hate",
}

# Create the numeric "labels" column used by the model.
train_df["labels"] = train_df[LABEL_COLUMN]
valid_df["labels"] = valid_df[LABEL_COLUMN]
test_df["labels"] = test_df[LABEL_COLUMN]

test_transliteration_df["labels"] = test_transliteration_df[LABEL_COLUMN]
test_codemix_df["labels"] = test_codemix_df[LABEL_COLUMN]
test_clean_sinhala_df["labels"] = test_clean_sinhala_df[LABEL_COLUMN]
test_misspell_df["labels"] = test_misspell_df[LABEL_COLUMN]
test_compound_df["labels"] = test_compound_df[LABEL_COLUMN]

print("label2id:", label2id)
print("id2label:", id2label)

In [ ]:
# ============================================================
# Convert pandas DataFrames to Hugging Face Dataset
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    valid_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset,
})

print(dataset)

test_transliteration_dataset = Dataset.from_pandas(
    test_transliteration_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_codemix_dataset = Dataset.from_pandas(
    test_codemix_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_clean_sinhala_dataset = Dataset.from_pandas(
    test_clean_sinhala_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_misspell_dataset = Dataset.from_pandas(
    test_misspell_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_compound_dataset = Dataset.from_pandas(
    test_compound_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

other_test_dataset = DatasetDict({
    "transliteration_test": test_transliteration_dataset,
    "codemix_test": test_codemix_dataset,
    "clean_sinhala_test": test_clean_sinhala_dataset,
    "misspell_test": test_misspell_dataset,
    "compound_test": test_compound_dataset
})

In [ ]:
# ============================================================
# Load original XLM-R tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
# ============================================================
# Inspect tokenization
# ============================================================

sample_texts = train_df[TEXT_COLUMN].head(5).tolist()

for text in sample_texts:

    tokens = tokenizer.tokenize(text)

    print("=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKENS:")
    print(tokens)

    print("\nNUMBER OF TOKENS:")
    print(len(tokens))

In [ ]:
# ============================================================
# Tokenize dataset
# ============================================================

def tokenize_function(examples):
    """
    Tokenize the input text using the original XLM-R tokenizer.

    truncation=True:
        Truncate examples longer than MAX_LENGTH.

    max_length=MAX_LENGTH:
        Maximum sequence length.

    We intentionally do not perform additional text cleaning here.
    """

    return tokenizer(
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
)

tokenized_test_dataset = other_test_dataset.map(
    tokenize_function,
    batched=True,
)

print(tokenized_dataset)
print(tokenized_test_dataset)

In [ ]:
# ============================================================
# Data collator
# ============================================================

# Instead of padding every example to 128 tokens during preprocessing, use dynamic padding.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created.")

In [ ]:
# ============================================================
# Load XLM-R sequence classification model
# ============================================================

model = XLMRobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print("Model loaded.")

In [ ]:
# ============================================================
# Freeze XLM-R encoder
# ============================================================

# First freeze EVERYTHING.
for param in model.parameters():
    param.requires_grad = False


# Now identify the classification head.
#
# XLM-R's classification model contains:
#     model.roberta  -> XLM-R encoder
#     model.classifier -> classification head
#
# We make only the classifier trainable.

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
# ============================================================
# Verify which parameters are trainable
# ============================================================

total_params = 0
trainable_params = 0

for name, param in model.named_parameters():

    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

        print("TRAINABLE:", name)


print("\n" + "=" * 60)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

In [ ]:
# ============================================================
# Evaluation metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits into predicted class IDs.
    predictions = np.argmax(logits, axis=-1)

    # Accuracy.
    accuracy = accuracy_score(
        labels,
        predictions
    )

    # Macro precision, recall and F1.
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # Weighted F1.
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

In [ ]:
# ============================================================
# Training configuration
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training.
    num_train_epochs=5,

    # Batch sizes.
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Since XLM-R is frozen, the classifier can tolerate
    # a higher learning rate than full-model fine-tuning.
    learning_rate=1e-3,

    # Regularization.
    weight_decay=0.01,

    # Evaluation.
    eval_strategy="epoch",

    # Save the model at each epoch.
    save_strategy="epoch",

    # Keep the best checkpoint.
    load_best_model_at_end=True,

    # Select the best model using Macro F1.
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Logging.
    logging_strategy="steps",
    logging_steps=50,

    # Use GPU if available.
    fp16=torch.cuda.is_available(),

    # Reproducibility.
    seed=SEED,

    # Reduce unnecessary checkpoint storage.
    save_total_limit=2,

    report_to="none",
)

In [ ]:
# ============================================================
# Create Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer created.")

In [ ]:
# ============================================================
# Train classification head
# ============================================================

train_result = trainer.train()

print("\nTraining completed.")

In [ ]:
# ============================================================
# Evaluate on validation set
# ============================================================

validation_results = trainer.evaluate(
    tokenized_dataset["validation"]
)

print("\nValidation results:")

for key, value in validation_results.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# Final test evaluation
# ============================================================

test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print("\nTest results:")

for key, value in test_results.items():
    print(f"{key}: {value}")


#### Transliteration Set Test

test_results_on_transliteration = trainer.evaluate(
    tokenized_test_dataset["transliteration_test"]
)
print("\nTest on Transliteration")
for key, value in test_results_on_transliteration.items():
    print(f"{key}: {value}")


#### Codemix Set Test

test_results_on_codemix = trainer.evaluate(
    tokenized_test_dataset["codemix_test"]
)
print("\nTest on Codemix")
for key, value in test_results_on_codemix.items():
    print(f"{key}: {value}")


#### Clean Sinhala Set Test

test_results_on_clean = trainer.evaluate(
    tokenized_test_dataset["clean_sinhala_test"]
)
print("\nTest on Clean Sinhala")
for key, value in test_results_on_clean.items():
    print(f"{key}: {value}")


#### Misspell Set Test

test_results_on_misspell = trainer.evaluate(
    tokenized_test_dataset["misspell_test"]
)
print("\nTest on Misspell")
for key, value in test_results_on_misspell.items():
    print(f"{key}: {value}")


#### Compound Set Test

test_results_on_compound = trainer.evaluate(
    tokenized_test_dataset["compound_test"]
)
print("\nTest on Compound")
for key, value in test_results_on_compound.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# 24. Detailed classification report
# ============================================================

predictions = trainer.predict(
    tokenized_dataset["test"]
)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 24. Detailed classification report - Transliteration
# ============================================================

predictions_tr = trainer.predict(
    tokenized_test_dataset["transliteration_test"]
)

predicted_labels_tr = np.argmax(
    predictions_tr.predictions,
    axis=-1
)

true_labels_tr = predictions_tr.label_ids

print(
    classification_report(
        true_labels_tr,
        predicted_labels_tr,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 24. Detailed classification report - CodeMix
# ============================================================

predictions_cm = trainer.predict(
    tokenized_test_dataset["codemix_test"]
)

predicted_labels_cm = np.argmax(
    predictions_cm.predictions,
    axis=-1
)

true_labels_cm = predictions_cm.label_ids

print(
    classification_report(
        true_labels_cm,
        predicted_labels_cm,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 24. Detailed classification report - Clean Sinhala
# ============================================================

predictions_cs = trainer.predict(
    tokenized_test_dataset["clean_sinhala_test"]
)

predicted_labels_cs = np.argmax(
    predictions_cs.predictions,
    axis=-1
)

true_labels_cs = predictions_cs.label_ids

print(
    classification_report(
        true_labels_cs,
        predicted_labels_cs,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 24. Detailed classification report - Misspell
# ============================================================

predictions_misspell = trainer.predict(
    tokenized_test_dataset["misspell_test"]
)

predicted_labels_misspell = np.argmax(
    predictions_misspell.predictions,
    axis=-1
)

true_labels_misspell = predictions_misspell.label_ids

print(
    classification_report(
        true_labels_misspell,
        predicted_labels_misspell,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 24. Detailed classification report - Compound
# ============================================================

predictions_compond = trainer.predict(
    tokenized_test_dataset["compound_test"]
)

predicted_labels_compond = np.argmax(
    predictions_compond.predictions,
    axis=-1
)

true_labels_compond = predictions_compond.label_ids

print(
    classification_report(
        true_labels_compond,
        predicted_labels_compond,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 25: Confusion matrix
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print("Confusion Matrix:")
print(cm)

In [ ]:
# ============================================================
# 26: Save trained model and tokenizer
# ============================================================

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:")
print(FINAL_MODEL_DIR)

In [ ]:
# ============================================================
# 27: Save experiment results
# ============================================================

experiment_results = {
    "experiment": "XLM-R Classification Head-Only Fine Tuning",
    "model": MODEL_NAME,
    "tokenizer": "Original XLM-R tokenizer",
    "encoder_trainable": False,
    "classification_head_trainable": True,
    "max_length": MAX_LENGTH,
    "seed": SEED,

    "accuracy": test_results.get("eval_accuracy"),
    "macro_precision": test_results.get("eval_macro_precision"),
    "macro_recall": test_results.get("eval_macro_recall"),
    "macro_f1": test_results.get("eval_macro_f1"),
    "weighted_f1": test_results.get("eval_weighted_f1"),

    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "trainable_percentage": (
        100 * trainable_params / total_params
    ),
}

results_df = pd.DataFrame([experiment_results])

display(results_df)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("Results saved.")

In [ ]:
# ============================================================
# 28: Test individual examples
# ============================================================

def predict_text(text):

    # Tokenize input text.
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Move inputs to the same device as the model.
    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Disable gradient calculation because we are predicting.
    with torch.no_grad():

        outputs = model(**inputs)

    # Convert logits to probabilities.
    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    return {
        "text": text,
        "prediction": id2label[predicted_id],
        "confidence": probabilities[0, predicted_id].item(),
    }


# Try a few samples from your test set.
for text in test_df[TEXT_COLUMN].head(10):

    result = predict_text(text)

    print("=" * 80)
    print(result)